# Semantic Clustering of Questions
This notebook uses a transformer model to semantically cluster user questions.

In [1]:
!pip install sentence-transformers pandas

  Using cached pandas-2.2.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (89 kB)
  Using cached numpy-2.2.4-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.3.2-py3-none-any.whl.metadata (11 kB)
  Using cached PyYAML-6.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached regex-2024.11.6-cp312-cp312-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached MarkupSafe-3.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (4.0 kB)
Using cached pandas-2.2.3-cp312-cp312-macosx_11_0_arm64.whl (11.4 MB)
Using cached numpy-2.2.4-cp312-cp312-macosx_14_0_arm64

In [8]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Load your CSV file
df = pd.read_csv("Support (1).csv")

In [9]:
questions = df['message'].dropna().tolist()


In [10]:
# Load the transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(questions, convert_to_tensor=True)

# Compute cosine similarity
cos_scores = util.pytorch_cos_sim(embeddings, embeddings)

In [12]:
# Perform clustering based on similarity threshold
threshold = 0.75
clusters = []
visited = set()

for i in range(len(questions)):
    if i in visited:
        continue
    cluster = [i]
    visited.add(i)
    for j in range(i + 1, len(questions)):
        if cos_scores[i][j] > threshold and j not in visited:
            cluster.append(j)
            visited.add(j)
    clusters.append(cluster)

In [13]:
# Extract representative questions
rep_questions = [questions[c[0]] for c in clusters]
pd.DataFrame(rep_questions, columns=["semantic_unique_questions"]).to_csv("semantic_unique_questions.csv", index=False)

# Save detailed cluster mapping
full_map = []
for c in clusters:
    full_map.append({
        "Representative Question": questions[c[0]],
        "Cluster Size": len(c),
        "Cluster Members": [questions[i] for i in c]
    })

pd.DataFrame(full_map).explode("Cluster Members").to_csv("semantic_clusters_detailed_2500.csv", index=False)